# Практическая работа №4. Выполнение задач обработки данных в распределённых вычислениях (Ву Дык Зуй - ИКБО-10-23)

## 0. Назначение практической работы

Цель работы - исследовать не только логический результат обработки данных в Apache Spark, но и физический механизм его получения. Исследуются связь преобразований и действий со Spark-стадиями и задачами, влияние числа партиций на параллелизм, появление перемешивания данных при широких преобразованиях, выбор стратегии JOIN, повторное вычисление по графу вычислений и эффект cache()/persist().

В предыдущей практической работе Spark использовался прежде всего как инструмент DataFrame-аналитики. В этой работе один и тот же набор операций рассматривается как вычислительный эксперимент. Студент должен уметь объяснить, почему два логически эквивалентных запроса могут иметь разный физический план и стоимость выполнения.

Численные значения времени выполнения в этой работе не являются эталонными: они зависят от процессора, объёма оперативной памяти, Java, версии Spark, фоновой нагрузки, файловой системы и прогрева JVM. Эталонным является направление анализа в котором предполагается фиксировать условия эксперимента (определенная машина со своими характеристиками), сравнивать варианты внутри одной сессии и связывать наблюдаемое время с физическим планом выполнения и Spark UI.

In [ ]:
print("Hello, World!")


In [ ]:
import pyspark
from IPython.display import Image


print(pyspark.__version__)


## 1. Стенд

В данном разделе фиксируются аппаратные и программные характеристики стенда, на котором выполняются все последующие эксперименты. Это необходимо для того, чтобы все замеры времени (benchmark) имели контекст. Абсолютное время выполнения Spark-задач сильно зависит от CPU, RAM, версии Java и настроек операционной системы.

In [ ]:
import platform
import multiprocessing

# Initialize SparkSession
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Practice4_SparkExecution")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


In [ ]:
# 1. Информация об ОС
print("=== Операционная система ===")
print(f"OC: {platform.system()} {platform.release()}")
print(f"Архитектура: {platform.machine()}")

# 2. Информация о процессоре
print("\n=== Аппаратное обеспечение ===")
print(f"Логических CPU (потоков): {multiprocessing.cpu_count()}")

# 3. Версии ПО
print("\n=== Версии ПО ===")
print(f"Python: {platform.python_version()}")
print(f"PySpark: {pyspark.__version__}")

# 4. Версии Java
java_version = spark.sparkContext._jvm.java.lang.System.getProperty('java.version')
print(f"Java: {java_version}")

# 5. Проверка конфигурации Spark
print("\n=== Конфигурация Spark ===")
print(f"spark.master: {spark.sparkContext.master}")
print(f"spark.default.parallelism: {spark.sparkContext.defaultParallelism}")
print(f"spark.sql.shuffle.partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"spark.sql.adaptive.enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"spark.sql.autoBroadcastJoinThreshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


## 2. Данные

На этом этапе мы извлекаем данные из SQLite в Parquet-формат. Parquet сохраняет схему и работает быстрее, чем CSV. После загрузки мы фиксируем базовые метрики (число строк) и определяем, на сколько исходных партиций (partitions) разбит каждый DataFrame при чтении.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd


DB_PATH = Path("ecommerce.sqlite")
STAGING = Path("spark_input")
PARQUET = Path("spark_input_parquet")

STAGING.mkdir(exist_ok=True)
PARQUET.mkdir(exist_ok=True)

# Needed tables (not all)
TABLES = [
    "geolocation", 
    "order_items", 
    "products", 
    "sellers", 
    "product_category_name_translation"
]

# 1. Extract from SQLite to CSV (if not already done)
if not (STAGING / "geolocation.csv").exists():
    with sqlite3.connect(DB_PATH) as con:
        for table in TABLES:
            out = STAGING / f"{table}.csv"
            # Use `chunksize` to avoid RAM overflow when handling the geolocation table (1 million rows)
            first = True
            total = 0
            for chunk in pd.read_sql_query(f"SELECT * FROM {table}", con, chunksize=100_000):
                chunk.to_csv(out, index=False, mode="w" if first else "a", header=first)
                total += len(chunk)
                first = False
            print(f"Staging: {table:40s} {total:>9d} rows")

# 2. Convert CSV to Parquet for faster reading by Spark
for name in TABLES:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(STAGING / f"{name}.csv"))
    )
    df.write.mode("overwrite").parquet(str(PARQUET / name))

print("✅ Все таблицы успешно переведены в Parquet.")


In [ ]:
# 1. Read data from Parquet
geo = spark.read.parquet(str(PARQUET / "geolocation"))
items = spark.read.parquet(str(PARQUET / "order_items"))
products = spark.read.parquet(str(PARQUET / "products"))
sellers = spark.read.parquet(str(PARQUET / "sellers"))
translations = spark.read.parquet(str(PARQUET / "product_category_name_translation"))

print("=== ИСХОДНЫЕ ДАННЫЕ ===")
print(f"geolocation:  {geo.count():>9d} строк, {geo.rdd.getNumPartitions()} partitions")
print(f"order_items:  {items.count():>9d} строк, {items.rdd.getNumPartitions()} partitions")
print(f"products:     {products.count():>9d} строк, {products.rdd.getNumPartitions()} partitions")
print(f"sellers:      {sellers.count():>9d} строк, {sellers.rdd.getNumPartitions()} partitions")
print(f"translations: {translations.count():>9d} строк, {translations.rdd.getNumPartitions()} partitions")

# Display schema of main table (order_items)
print("\n=== СХЕМА ORDER_ITEMS ===")
items.printSchema()


### Выводы

Все таблицы успешно прочитаны из Parquet. Количество исходных партиций зависит от количества файлов, созданных при записи. Так как мы не меняли `repartition()` при записи в Parquet, Spark разбил данные на партиции по умолчанию, исходя из размера файлов.

## 3. Lazy evaluation

В этом разделе мы экспериментально докажем принцип ленивых вычислений в Spark. Мы создадим цепочку преобразований (Transformations), но не будем вызывать действия (Actions). Затем мы вызовем `count()` и увидим, как в Spark UI появляется новый Job.

In [ ]:
from pyspark.sql import functions as F
from time import perf_counter


# Set Job groups name for easier search in Spark UI
sc = spark.sparkContext
sc.setJobGroup("P5_LAZY", "Lazy evaluation demo")

# Create 3 Transformations
geo_prepared = (
    geo
    .select("geolocation_zip_code_prefix", "geolocation_state", "geolocation_lat", "geolocation_lng")
    .filter(F.col("geolocation_state").isNotNull())
    .withColumn("abs_lat", F.abs(F.col("geolocation_lat")))
)

# Check returned object type
print("Тип объекта:", type(geo_prepared).__name__)
print("Данные еще НЕ вычислены, план только построен.")
display(Image(filename='images/spark1.png'))


In [ ]:
# Call Action count(). 
# That's when Spark really read Parquet file, filter and calculate.
t0 = perf_counter()
row_count = geo_prepared.count()
elapsed = perf_counter() - t0

print(f"Количество строк: {row_count}")
print(f"Время выполнения: {elapsed:.3f} сек.")


In [ ]:
display(Image(filename='images/spark2.png'))


### Интерпретация результатов

1. **До вызова `count()`:** Объект `geo_prepared` был создан мгновенно (за доли секунды). Это произошло потому, что Spark не читал миллион строк из файла. Он просто запомнил логический план (Logical Plan): "Взять файл -> выбрать 4 столбца -> отфильтровать пустые -> добавить новый столбец". На этом этапе не было задействовано ни одного ядра процессора.
2. **После вызова `count()`:** Время выполнения резко возросло, так как Spark запустил реальный физический план. Мы видим в Spark UI, что был создан Job, разделенный на Stages и Tasks. Только в этот момент данные были прочитаны с диска и обработаны.

### Выводы

Это доказывает, что время создания DataFrame (написания кода) ничего не говорит о стоимости вычислений. Стоимость возникает только на этапе Action.

## 4. Shuffle

В отличие от `select()` или `filter()`, которые обрабатывают данные локально в каждой партиции (narrow transformations), операция `groupBy()` требует собрать строки с одинаковыми ключами вместе. Это вызывает **Shuffle** — физическую передачу данных между узлами/партициями. Это самая дорогая операция в Spark.

In [ ]:
# Set Job name for searching in Spark UI
sc.setJobGroup("P5_GROUPBY", "Shuffle: groupBy seller_id")

# Calculate sales statistics by individual seller with seller_id
seller_stats = (
    items
    .groupBy("seller_id")
    .agg(
        F.count("*").alias("items_count"),
        F.round(F.sum("price"), 2).alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )
    .orderBy(F.desc("revenue"))
)

print("План выполнения (Physical Plan):")
seller_stats.explain("formatted")


In [ ]:
# Run Action show() and Spark will really do Shuffle
print("Топ-5 продавцов по выручке:")
seller_stats.show(5, truncate=False)


### Интерпретация результатов

1. **Exchange = Shuffle:** Оператор `Exchange hashpartitioning(seller_id, 8)` в плане подтверждает, что Spark перераспределяет данные. Символ `8` означает, что результирующих партиций будет 8 (задано в `spark.sql.shuffle.partitions`).
2. **Две стадии:** В Spark UI мы видим, что Job разбился на 2 Stages. Это классический признак Shuffle. Stage 1 выполняет частичную агрегацию (`partial_sum`) в каждой партиции и записывает результат на диск (Shuffle Write). Stage 2 читает эти промежуточные данные (Shuffle Read) и выполняет финальную агрегацию.
3. **Цена Shuffle:** В отличие от `filter()` (который просто отбрасывает строки), `groupBy()` требует чтения/записи на диск и сетевой передачи (даже в режиме `local[*]` это эмуляция сети). Поэтому `groupBy()` всегда работает медленнее, чем `select()` или `where()`.

In [ ]:
display(Image(filename='images/spark3.png'))


### Выводы

Spark всегда старается выполнить как можно больше работы *до* Shuffle (например, отфильтровать лишние строки), чтобы уменьшить объем данных, которые придется перемещать по сети.

## 5. JOIN

В этом разделе мы выполним логически одинаковый JOIN между таблицей `order_items` и `products` двумя разными способами. Сначала мы принудительно вызовем `Sort Merge Join` (сортировка и слияние), а затем `Broadcast Hash Join` (широковещательный хэш). Результат строк будет одинаковым, но физические планы и нагрузка на сеть будут разными.

In [ ]:
sc.setJobGroup("P5_SMJ", "Sort Merge Join: items x products")

# Force Spark to use merge strategy
smj = (
    items.hint("merge")
    .join(products.hint("merge"), on="product_id", how="inner")
)

print("=== PHYSICAL PLAN: SORT MERGE JOIN ===")
smj.explain("formatted")

# Calculate time and display
t0 = perf_counter()
smj_count = smj.count()
elapsed_smj = perf_counter() - t0
print(f"\nSMJ Rows: {smj_count}")
print(f"SMJ Time: {elapsed_smj:.3f} s")


In [ ]:
sc.setJobGroup("P5_BHJ", "Broadcast Hash Join: items x products")

# Dùng F.broadcast để ép Spark gửi bảng products đến tất cả các node
bhj = (
    items
    .join(F.broadcast(products), on="product_id", how="inner")
)

print("=== PHYSICAL PLAN: BROADCAST HASH JOIN ===")
bhj.explain("formatted")

# Замеряем время
t0 = perf_counter()
bhj_count = bhj.count()
elapsed_bhj = perf_counter() - t0
print(f"\nBHJ Rows: {bhj_count}")
print(f"BHJ Time: {elapsed_bhj:.3f} s")


In [ ]:
print(f"{'Показатель':<30} | {'Sort Merge Join':<20} | {'Broadcast Hash Join':<20}")
print("-" * 75)
print(f"{'Строк результата':<30} | {smj_count:<20} | {bhj_count:<20}")
print(f"{'Время (сек)':<30} | {elapsed_smj:<20.3f} | {elapsed_bhj:<20.3f}")


### Интерпретация результатов JOIN

1. **Логическая эквивалентность:** Оба запроса вернули абсолютно одинаковое количество строк (112 650). Логика соединения не изменилась, изменилась только физическая реализация.
2. **Sort Merge Join (SMJ):** В плане мы видим два оператора `Exchange`. Обе таблицы (и большая `order_items`, и средняя `products`) были перемешаны (Shuffle) и отсортированы. Это тяжелая операция, требующая много сетевого трафика и памяти.
3. **Broadcast Hash Join (BHJ):** В плане мы видим `BroadcastExchange` только для таблицы `products`. Spark взял таблицу `products` (около 32 000 строк), отправил её копию в каждую партицию таблицы `items`, и выполнил JOIN прямо на месте (in-memory). Это сэкономило время, так как большую таблицу `order_items` не пришлось перемешивать.
4. **Почему время может отличаться?** В локальном режиме разница во времени может быть не такой колоссальной, как в реальном кластере, но она есть. В кластере BHJ сэкономит минуты (или часы), исключив Shuffle огромной таблицы.

### Выводы

Broadcast Hash Join идеален, когда одна таблица маленькая (помещается в память), а вторая — большая. Если обе таблицы гигантские, Broadcast упадёт с ошибкой `OutOfMemory`, и придётся использовать Sort Merge Join.

## 6. Cache

В этом разделе мы создадим сложный DataFrame, включающий JOIN и вычисляемые столбцы. Мы выполним два разных действия (Actions) над этим DataFrame. Сначала мы замерим время выполнения без кэширования, затем применим `persist(MEMORY_AND_DISK)` и замерим время снова, чтобы оценить эффект материализации.

In [ ]:
# Clean old cache if exists
spark.catalog.clearCache()

# Select only needed columns for JOIN
products_small = products.select("product_id", "product_category_name")
sellers_small = sellers.select("seller_id", "seller_state")

# Create complex DataFrame (JOIN + calculated column)
joined = (
    items
    .join(F.broadcast(products_small), "product_id", "left")
    .join(F.broadcast(sellers_small), "seller_id", "left")
    .withColumn("gross_value", F.col("price") + F.col("freight_value"))
)

# Functions for performing two different aggregations
def action_category(df):
    return df.groupBy("product_category_name").agg(F.sum("gross_value").alias("gross")).agg(F.sum("gross").alias("checksum")).first()[0]

def action_state(df):
    return df.groupBy("seller_state").agg(F.sum("gross_value").alias("gross")).agg(F.sum("gross").alias("checksum")).first()[0]

print("DataFrame создан. Проверка is_cached:", joined.is_cached)


In [ ]:
sc.setJobGroup("P5_NOCACHE", "Два действия без сохранения")

t0 = perf_counter()
_ = action_category(joined)
t1 = perf_counter()
_ = action_state(joined)
t2 = perf_counter()

print(f"Action 1 (Category): {t1 - t0:.3f} s")
print(f"Action 2 (State):    {t2 - t1:.3f} s")
print("is_cached:", joined.is_cached)


In [ ]:
from pyspark import StorageLevel


# Add persist (Save in memory and disk)
cached = joined.persist(StorageLevel.MEMORY_AND_DISK)
print("После persist:", cached.is_cached)

# Materialization: The first Action fills the cache
t0 = perf_counter()
materialized_rows = cached.count()
t1 = perf_counter()

# Subsequent actions use an already cached DataFrame.
_ = action_category(cached)
t2 = perf_counter()
_ = action_state(cached)
t3 = perf_counter()

print(f"Materialization (first count): {t1 - t0:.3f} s")
print(f"Cached Action 1 (Category):    {t2 - t1:.3f} s")
print(f"Cached Action 2 (State):       {t3 - t2:.3f} s")

# Free cache
cached.unpersist(blocking=True)
print("После unpersist:", cached.is_cached)


### Интерпретация результатов Cache

1. **Без кэша:** Второе действие (`action_state`) потребовало почти столько же времени, сколько первое. Это доказывает принцип повторного вычисления lineage. Так как DataFrame не был сохранен, Spark каждый раз заново читал Parquet и выполнял JOIN.
2. **Материализация:** Обратите внимание на время выполнения `cached.count()`. Это время больше или равно времени первого действия. `persist()` — это ленивая операция. Она не вычисляет данные до тех пор, пока не будет вызван первый Action. Поэтому первый `count()` фактически заполняет кэш.
3. **Повторные действия с кэшем:** Время выполнения `action_category` и `action_state` после материализации резко сократилось. Spark не выполняет JOIN и не читает диск заново, он просто берет готовые данные из оперативной памяти/локального диска.
4. **Завершение:** После вызова `unpersist()` кэш очищается, чтобы не занимать память для следующих задач.

### Выводы

`cache()` / `persist()` окупается только при **повторном использовании** одного и того же DataFrame. Если DataFrame используется один раз, кэширование только замедлит работу из-за накладных расходов на сохранение данных.

## 7. Benchmark

В этом разделе мы проведем контролируемый бенчмарк. Мы создадим синтетический DataFrame с 2 миллионами строк и будем выполнять агрегацию, меняя только количество входных партиций (от 1 до 32). Для надежности каждый замер выполняется 3 раза, а итоговым результатом считается медиана (median), так как одиночный замер может быть искажен прогревом JVM или фоновыми процессами.

In [ ]:
from statistics import median


N = 2_000_000
P_VALUES = [1, 2, 4, 8, 16, 32]

# Let’s ensure that the number of shuffle partitions is set to 8,
# for the sake of a clean experiment.
spark.conf.set("spark.sql.shuffle.partitions", "8")

def run_benchmark(p, repeats=3):
    times = []
    checksum = None
    for _ in range(repeats):
        # Create a DataFrame with a specified number of partitions
        df = (
            spark.range(0, N, 1, numPartitions=p)
            .withColumn("key", (F.col("id") % 1000).cast("int"))
            .withColumn("value", F.sqrt(F.col("id") + 1.0))
        )
        
        t0 = perf_counter()
        # Performing aggregation (Action)
        checksum = (
            df.groupBy("key")
            .agg(F.sum("value").alias("s"))
            .agg(F.sum("s").alias("checksum"))
            .first()[0]
        )
        times.append(perf_counter() - t0)
        
    return times, median(times), checksum

# Warm-up to allow the JVM and JIT compiler to optimize the code
# before actual measurements are taken
print("Прогрев...")
_ = run_benchmark(4, repeats=1)
print("Прогрев завершен.\n")


In [ ]:
print(f"{'p':<3} | {'Run 1 (s)':<10} | {'Run 2 (s)':<10} | {'Run 3 (s)':<10} | {'Median (s)':<10}")
print("-" * 55)

results = []
for p in P_VALUES:
    times, med, checksum = run_benchmark(p, repeats=3)
    results.append(med)
    print(f"{p:<3} | {times[0]:<10.3f} | {times[1]:<10.3f} | {times[2]:<10.3f} | {med:<10.3f}")


### Интерпретация результатов бенчмарка

Анализ медианных значений времени выполнения показывает следующую динамику:

1. **Влияние прогрева JVM (Run 1):** В первых запусках для каждого `p` время всегда значительно выше (от 0.47 до 0.67 с). Это связано с тем, что Spark только инициализирует задачи, а виртуальная машина Java (JVM) выполняет JIT-компиляцию кода. Начиная со второго запуска, время стабилизируется.
2. **Зона минимального времени (p = 1, 4, 8):** Вопреки классической теории, где `p=1` должно быть самым медленным, в нашем эксперименте медиана для `p=1` составила **0.180 с**, что является лучшим результатом. Это объясняется тем, что для синтетического датафрейма малого размера (2 млн строк) накладные расходы на координацию задач (scheduling overhead) при малом числе партиций оказались ниже, чем затраты на параллелизацию. Значения `p=4` (0.274 с) и `p=8` (0.263 с) показали стабильно близкие результаты, так как смогли задействовать несколько ядер CPU.
3. **Деградация при избытке партиций (p = 16, 32):** При увеличении числа партиций до 16 и 32 медианное время выросло до **0.393 с** и **0.421 с** соответственно. Это наглядная демонстрация **overhead**. Слишком большое количество мелких задач приводит к тому, что Spark тратит больше времени на управление задачами (Task Scheduler), сериализацию данных и переключение контекста, чем на сами вычисления.
4. **Влияние Shuffle (фиксированный параметр 8):** Поскольку мы зафиксировали `spark.sql.shuffle.partitions = 8`, то при `p = 16` и `p = 32` данные после группировки всё равно сжимались до 8 партиций. Это создало дополнительную нагрузку на этап Exchange (перераспределения).

### Выводы

В локальном режиме (`local[*]`) зависимость времени от числа партиций не является строго линейной. Для данного стенда и объема данных оптимальным оказался режим с малым числом партиций (p=1 или p=4), где отсутствовали накладные расходы на координацию. Однако на реальных Big Data кластерах с гигабайтами данных ситуация изменится: там `p=1` вызовет OOM (нехватку памяти), и оптимальным будет диапазон `p`, кратный количеству ядер CPU.

## 8. Small files

При записи DataFrame в формате Parquet Spark создает отдельный файл для каждой выходной партиции. Если партиций слишком много, генерируется множество маленьких файлов. Это создает overhead (накладные расходы) при чтении, так как системе приходится открывать и закрывать тысячи файлов. В этом разделе мы запишем один и тот же датафрейм с разным числом партиций и измерим средний размер файла.

In [ ]:
from pathlib import Path


# Directory path containing the results
OUT_SMALL = Path("small_files_experiment")
OUT_SMALL.mkdir(exist_ok=True)

def write_and_measure(p):
    path = OUT_SMALL / f"p_{p}"
    
    # Write data with p partitions
    (
        items.repartition(p)
        .write.mode("overwrite")
        .parquet(str(path))
    )
    
    # Measure file size and quantity
    files = list(path.glob("part-*.parquet"))
    sizes = [f.stat().st_size for f in files]
    
    total_mib = sum(sizes) / (1024**2)
    avg_kib = (sum(sizes) / len(sizes) / 1024) if sizes else 0
    
    return len(files), total_mib, avg_kib


In [ ]:
print(f"{'Партиций (p)':<12} | {'Файлов':<8} | {'Общий размер (MiB)':<20} | {'Средний файл (KiB)':<20}")
print("-" * 70)

P_VALUES_SMALL_FILES = [1, 4, 16, 64]

for p in P_VALUES_SMALL_FILES:
    num_files, total_mib, avg_kib = write_and_measure(p)
    print(f"{p:<12} | {num_files:<8} | {total_mib:<20.2f} | {avg_kib:<20.1f}")


### Интерпретация результатов

1. **Связь partitions и файлов:** Результаты эксперимента подтверждают, что количество выходных `part-*.parquet` файлов напрямую зависит от числа партиций. `repartition(64)` создает 64 отдельные задачи записи, и каждая задача пишет свой собственный файл.
2. **Проблема малых файлов (Small Files Problem):** При `p=64` мы получаем множество маленьких файлов. Хотя суммарный объем данных тот же, наличие 64 файлов вместо 1 означает, что при последующем чтении Spark придется выполнить 64 операции открытия файла, прочитать 64 блока метаданных из файловой системы. На реальных кластерах (например, HDFS или S3) это вызывает огромную задержку. Такое чтение работает гораздо медленнее, чем чтение 1 большого файла.
3. **Почему `coalesce(1)` — не всегда панацея:** Если мы сделаем `coalesce(1)`, мы получим 1 файл, что отлично для маленьких таблиц. Но если наш DataFrame весит 100 ГБ, `coalesce(1)` заставит весь этот объем записывать **только один CPU core**, пока остальные будут простаивать. Поэтому для больших данных файлы нужно группировать оптимально (например, по 128 МБ на файл).

### Выводы

Эксперимент наглядно продемонстрировал прямую связь между числом выходных партиций (`p`) и количеством создаваемых `part-файлов`. При увеличении `p` от 1 до 64 общий объем данных остался практически неизменным, однако количество файлов выросло до 64, а их средний размер кратно уменьшился. 

Это доказывает существование **проблемы маленьких файлов (Small Files Problem)**. Хранение большого числа мелких файлов создает значительные накладные расходы (overhead) при последующем чтении данных: файловой системе и драйверу Spark требуется гораздо больше времени на открытие файлов, чтение метаданных и планирование задач, чем на обработку самих данных. 

Следовательно, количество партиций при записи нельзя выбирать произвольно. Хотя `coalesce(1)` позволяет получить один файл, для больших объемов данных это решение неприемлемо, так как оно уничтожает параллелизм записи (все данные пишет одна задача). Оптимальной стратегией является балансировка: размер каждого `part-файла` должен составлять десятки или сотни мегабайт, чтобы минимизировать накладные расходы файловой системы без потери производительности распределенной записи.

## 9. Итоговые выводы

На основе проведённых экспериментов с физическим планом выполнения и Spark UI можно сформулировать следующие выводы:

1. **Ленивые вычисления (Lazy Evaluation):** Построение цепочки трансформаций (`select`, `filter`, `withColumn`) не запускает вычисления. Spark лишь формирует логический план. Фактическое чтение данных и распределённая обработка начинаются только при вызове Action (`count()`, `show()`), что подтверждается появлением Job в Spark UI.
2. **Связь partitions и параллелизма:** Число задач (Tasks) в Stage напрямую зависит от числа партиций. При малом числе партиций (например, `p=1`) вычисления выполняются последовательно, и ядра CPU простаивают. Однако слишком большое число партиций (например, `p=32`) приводит к накладным расходам на планирование задач, что увеличивает общее время выполнения. 
3. **Стоимость Shuffle:** Широкие трансформации (такие как `groupBy`) вызывают перераспределение данных (Shuffle). Это проявляется в плане как оператор `Exchange` и разделяет Job на несколько Stages. Shuffle — самая дорогая операция, требующая записи на диск и сетевой передачи, поэтому Spark старается выполнить частичную агрегацию до Shuffle, чтобы уменьшить объём передаваемых данных.
4. **Стратегии JOIN:** Логически одинаковые запросы могут иметь разный физический план. Sort Merge Join требует перемешивания и сортировки обеих таблиц. В то же время Broadcast Hash Join (при котором маленькая таблица копируется на все узлы) полностью исключает Shuffle для большой таблицы, что делает его значительно быстрее при правильном выборе таблицы для broadcast.
5. **Эффективность Cache/Persist:** Если DataFrame используется многократно, кэширование (`persist()`) позволяет избежать повторного вычисления lineage (чтения файлов и JOIN). Время первого Action после `persist()` включает материализацию (заполнение кэша), но все последующие Actions выполняются значительно быстрее. При однократном использовании кэш не окупается.
6. **Проблема маленьких файлов:** Число выходных файлов при записи (`write.parquet`) равно числу партиций. Слишком большое число партиций порождает множество мелких файлов, что создаёт огромные накладные расходы при последующем чтении. Использование `coalesce(1)` решает проблему для малых таблиц, но убивает параллелизм на больших данных.

### Ограничения измерений

1. **Локальный режим (`local[*]`):** Все замеры производились на одной машине. В физическом плане здесь эмулируется сетевой обмен (Shuffle), но фактически обмен идёт через оперативную память одного компьютера. На реальном кластере накладные расходы на Shuffle и передачу данных по сети будут значительно выше, а абсолютное время выполнения непредсказуемо возрастёт.
2. **Влияние прогрева JVM и фоновых процессов:** Замеры времени в Spark зависят от JIT-компиляции, кэша операционной системы и фоновой нагрузки. Именно поэтому для оценки использовалась медиана трёх запусков (а не единичный замер), однако даже медиана на локальном стенде даёт лишь качественное представление о трендах, а не универсальные эталонные цифры.

In [ ]:
spark.stop()
